# 🧠 TS2Vec — Time Series Embeddings for Road Surface Classification

> **TS2Vec** is a self-supervised contrastive learning framework that learns universal representations of time series.  
> No labels needed. Works by contrasting different temporal views of the same signal.

---

### How TS2Vec Works (intuition)
```
Window A (gravel segment)
   ├── View 1: random crop + timestamp masking  ─┐
   └── View 2: different crop + masking          ─┴─► Encoder ──► embeddings
                                                         ↑
                                              Trained to make views of SAME
                                              segment similar, different segments
                                              dissimilar  (contrastive loss)
```

### Table of Contents
1. [Install & Imports](#1)
2. [How TS2Vec Differs from Other Approaches](#2)
3. [Data Preparation](#3)
4. [Train TS2Vec Model](#4)
5. [Generate Embeddings (3 Modes)](#5)
6. [Post-Processing & Clustering](#6)
7. [Evaluation & Visualization](#7)
8. [Fine-Tuning Tips](#8)
9. [Full Reusable Pipeline Function](#9)


## 1. Install & Imports <a id='1'></a>

In [ ]:
# Install ts2vec and dependencies
# !pip install ts2vec torch numpy scikit-learn matplotlib umap-learn

# ts2vec is also available directly from GitHub:
# !pip install git+https://github.com/yuezhihan/ts2vec.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device    : {DEVICE}')

# TS2Vec
try:
    from ts2vec import TS2Vec
    print('✅ TS2Vec imported successfully')
except ImportError:
    print('❌ ts2vec not found. Run: pip install ts2vec')

# Supporting libraries
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

try:
    from umap import UMAP
    UMAP_AVAILABLE = True
    print('✅ UMAP available')
except ImportError:
    from sklearn.manifold import TSNE
    UMAP_AVAILABLE = False
    print('⚠️  UMAP not found — will use t-SNE')


## 2. How TS2Vec Differs from Other Approaches <a id='2'></a>

| Feature | Hand-Crafted | ROCKET | **TS2Vec** |
|---------|-------------|--------|------------|
| Requires training | ❌ No | ❌ No | ✅ Yes (self-supervised) |
| Needs labels | ❌ No | ❌ No | ❌ No |
| Learns from your data | ❌ No | ❌ No | ✅ Yes |
| Captures temporal context | ⚠️ Partial | ⚠️ Partial | ✅ Full |
| Multi-scale representation | ❌ No | ❌ No | ✅ Yes |
| Best dataset size | Any | Medium+ | Large (>5k windows) |
| Embedding quality | Good | Better | Best |

### TS2Vec Output Modes
TS2Vec can encode at **three levels of granularity** — this is its key advantage:

```
Input window (3000 timesteps × 3 axes)
       │
  TS2Vec Encoder
       │
       ├── Instance-level  → 1 vector per window   (shape: N × output_dims)
       │                      ✅ Best for clustering windows = road surfaces
       │
       ├── Timestamp-level → 1 vector per timestep (shape: N × T × output_dims)
       │                      Use for: anomaly detection at specific moments
       │
       └── Multi-scale     → hierarchical pooling  (shape: N × output_dims)
                              Use for: capturing both fine & coarse surface patterns
```

> **For road surface clustering → use Instance-level encoding**


## 3. Data Preparation <a id='3'></a>

### TS2Vec Input Format
```
Shape: (N, T, C)
  N = number of windows / samples
  T = timesteps per window  (e.g. 3000 for 3s @ 1kHz)
  C = number of channels    (3 for X, Y, Z axes)
```
This is different from sklearn convention `(N, features)` — TS2Vec works on **raw time series**, not pre-extracted features.


In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
FS             = 1000     # Sampling frequency (Hz)
WINDOW_SEC     = 3.0      # Window length in seconds
OVERLAP        = 0.5      # 50% overlap
OUTPUT_DIMS    = 320      # TS2Vec embedding size (default 320, try 128 for smaller data)
N_EPOCHS       = 50       # Training epochs (increase for better quality)
BATCH_SIZE     = 8        # Reduce if OOM error
N_CLUSTERS     = 4        # Number of road surface types

WINDOW_LEN = int(FS * WINDOW_SEC)
print(f'Window length : {WINDOW_LEN} timesteps  ({WINDOW_SEC}s @ {FS}Hz)')


# ── SYNTHETIC DEMO DATA (replace with your CSV) ───────────────────────────
# To use your real data:
#   df = pd.read_csv('your_data.csv')
#   DATA = df[['acc_x','acc_y','acc_z']].values.astype(np.float32)  # (N_total, 3)
#   GT_LABELS = df['label'].values if 'label' in df.columns else None

np.random.seed(42)
dur = 30   # seconds per surface
surface_profiles = {
    'asphalt':     (0.05, [0.60, 0.30, 0.05, 0.03, 0.02]),
    'gravel':      (0.30, [0.10, 0.20, 0.35, 0.25, 0.10]),
    'cobblestone': (0.20, [0.20, 0.50, 0.20, 0.06, 0.04]),
    'dirt':        (0.25, [0.15, 0.20, 0.30, 0.20, 0.15]),
}
band_centers = [5, 20, 45, 80, 125]

segments, labels_list = [], []
for surface, (noise_std, band_weights) in surface_profiles.items():
    n  = FS * dur
    t  = np.linspace(0, dur, n)
    sig = np.zeros((n, 3))
    for amp, freq in zip(band_weights, band_centers):
        for ax in range(3):
            sig[:, ax] += amp * np.sin(2 * np.pi * freq * t + np.random.uniform(0, 2*np.pi))
    sig += np.random.randn(n, 3) * noise_std
    sig = (sig - sig.mean(0)) / (sig.std(0) + 1e-8)   # z-score (already done in your data)
    segments.append(sig)
    labels_list.extend([surface] * n)

DATA      = np.vstack(segments).astype(np.float32)   # (N_total, 3)
GT_LABELS = np.array(labels_list)
print(f'Raw data shape : {DATA.shape}')


# ── WINDOWING ──────────────────────────────────────────────────────────────
def create_windows(data, window_len, step, gt_labels=None):
    windows, win_gt = [], []
    for start in range(0, len(data) - window_len + 1, step):
        windows.append(data[start:start + window_len])   # (T, 3)
        if gt_labels is not None:
            seg = gt_labels[start:start + window_len]
            u, c = np.unique(seg, return_counts=True)
            win_gt.append(u[np.argmax(c)])
    return np.array(windows), (win_gt if gt_labels is not None else None)

step    = int(WINDOW_LEN * (1 - OVERLAP))
WINDOWS, WINDOW_GT = create_windows(DATA, WINDOW_LEN, step, GT_LABELS)

# TS2Vec input: (N, T, C)  — already this shape from windowing
print(f'\nTS2Vec input shape : {WINDOWS.shape}')
print(f'  N={WINDOWS.shape[0]} windows | T={WINDOWS.shape[1]} timesteps | C={WINDOWS.shape[2]} axes')
if WINDOW_GT:
    from collections import Counter
    print(f'Windows per surface: {dict(Counter(WINDOW_GT))}')


## 4. Train TS2Vec Model <a id='4'></a>

### Key Parameters Explained

| Parameter | What it does | Suggested value |
|-----------|-------------|----------------|
| `input_dims` | Number of axes (channels) | 3 (X, Y, Z) |
| `output_dims` | Embedding size | 128–320 |
| `hidden_dims` | Internal encoder width | 64 (default) |
| `depth` | Encoder depth (dilated conv layers) | 10 (default) |
| `lr` | Learning rate | 0.001 |
| `batch_size` | Windows per batch | 8–32 |
| `temporal_unit` | Smallest temporal scale for contrastive loss | 0 |

> **No labels are used during training** — TS2Vec learns by contrasting augmented views of the same window.


In [ ]:
model = TS2Vec(
    input_dims   = WINDOWS.shape[2],   # 3 axes (X, Y, Z)
    output_dims  = OUTPUT_DIMS,        # embedding dimension
    hidden_dims  = 64,                 # encoder hidden size
    depth        = 10,                 # dilated conv depth
    device       = DEVICE,
    lr           = 0.001,
    batch_size   = BATCH_SIZE,
)

print('TS2Vec model created')
print(f'  Input dims  : {WINDOWS.shape[2]}')
print(f'  Output dims : {OUTPUT_DIMS}')
print(f'  Device      : {DEVICE}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Epochs      : {N_EPOCHS}')
print()

# ── TRAIN ──────────────────────────────────────────────────────────────────
# Input to fit(): (N, T, C)  — numpy float array
# No labels needed — purely self-supervised
print('Training TS2Vec... (self-supervised, no labels needed)')

loss_log = model.fit(
    WINDOWS,              # shape: (N, T, C)
    n_epochs   = N_EPOCHS,
    verbose    = True,    # prints loss per epoch
)

print('\n✅ Training complete!')

# Plot training loss
plt.figure(figsize=(8, 3))
plt.plot(loss_log, color='steelblue', lw=2)
plt.title('TS2Vec Training Loss', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Contrastive Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── SAVE & LOAD MODEL ─────────────────────────────────────────────────────
# Save trained model so you don't have to retrain
model.save('ts2vec_road_surface.pkl')
print('Model saved → ts2vec_road_surface.pkl')

# To reload later:
# model = TS2Vec(input_dims=3, output_dims=OUTPUT_DIMS, device=DEVICE)
# model.load('ts2vec_road_surface.pkl')
# print('Model loaded ✅')


## 5. Generate Embeddings — 3 Modes <a id='5'></a>

TS2Vec supports three encoding modes. **Use instance-level for road surface clustering.**

```
encode(X, encoding_window=...):

  encoding_window = 'full_series'   → Instance-level  ✅ road surface clustering
  encoding_window = None            → Timestamp-level    anomaly / event detection  
  encoding_window = k  (int)        → Sliding window     local pattern detection
```


In [ ]:
# ── MODE 1: Instance-level (ONE vector per window) ───────────────────────
# Best for: clustering whole windows into surface types
# Output : (N, output_dims)

EMB_INSTANCE = model.encode(
    WINDOWS,
    encoding_window = 'full_series',   # pool over entire window
    batch_size       = BATCH_SIZE,
)
print(f'Instance-level embeddings : {EMB_INSTANCE.shape}')
print(f'  → {EMB_INSTANCE.shape[0]} windows  ×  {EMB_INSTANCE.shape[1]} dims')
print(f'  → Use this for road surface clustering ✅')


In [ ]:
# ── MODE 2: Timestamp-level (one vector per timestep) ────────────────────
# Best for: detecting exact moment when surface changes
# Output : (N, T, output_dims)

EMB_TIMESTAMP = model.encode(
    WINDOWS,
    encoding_window = None,   # no pooling — keep all timesteps
    batch_size       = BATCH_SIZE,
)
print(f'Timestamp-level embeddings : {EMB_TIMESTAMP.shape}')
print(f'  → {EMB_TIMESTAMP.shape[0]} windows  ×  {EMB_TIMESTAMP.shape[1]} timesteps  ×  {EMB_TIMESTAMP.shape[2]} dims')
print(f'  → Use this for surface transition detection')

# To get window-level embedding from timestamp-level: pool over time axis
EMB_FROM_TIMESTAMP = EMB_TIMESTAMP.mean(axis=1)   # (N, output_dims)
print(f'\nMean-pooled from timestamp : {EMB_FROM_TIMESTAMP.shape}')


In [ ]:
# ── MODE 3: Multi-scale sliding window ───────────────────────────────────
# Best for: capturing both fine texture (gravel pebbles) and
#           coarse pattern (cobblestone periodicity)
# Output : (N, output_dims)

# sliding window size = fraction of total window length
SLIDING_K = WINDOW_LEN // 4   # 25% of window = 750 timesteps for 3s window

EMB_MULTISCALE = model.encode(
    WINDOWS,
    encoding_window = SLIDING_K,
    batch_size       = BATCH_SIZE,
)
print(f'Multi-scale embeddings : {EMB_MULTISCALE.shape}')
print(f'  Sliding window size  : {SLIDING_K} timesteps ({SLIDING_K/FS:.2f}s)')

# ── Summary ───────────────────────────────────────────────────────────────
print('\n' + '─'*50)
print('Embedding shape summary:')
print(f'  Instance-level  : {EMB_INSTANCE.shape}   ← use for clustering')
print(f'  Timestamp-level : {EMB_TIMESTAMP.shape}')
print(f'  Multi-scale     : {EMB_MULTISCALE.shape}')


## 6. Post-Processing & Clustering <a id='6'></a>

In [ ]:
# ── Use instance-level for clustering ─────────────────────────────────────
EMB = EMB_INSTANCE.copy()

# Handle any NaN/Inf
EMB = np.nan_to_num(EMB, nan=0.0, posinf=0.0, neginf=0.0)

# L2 normalize (enables cosine distance)
EMB_NORM = normalize(EMB, norm='l2')

# PCA to reduce noise (TS2Vec at 320-dims often has redundancy)
pca = PCA(n_components=0.95, random_state=42)
EMB_PCA = pca.fit_transform(EMB_NORM)
print(f'Dimensionality: {EMB.shape[1]} → L2 norm → PCA → {EMB_PCA.shape[1]} dims')

# ── Find optimal K ────────────────────────────────────────────────────────
print('\nSearching optimal K...')
ks, sils, dbs = [], [], []
for k in range(2, 9):
    km  = KMeans(n_clusters=k, n_init=20, random_state=42)
    lbl = km.fit_predict(EMB_PCA)
    ks.append(k)
    sils.append(silhouette_score(EMB_PCA, lbl))
    dbs.append(davies_bouldin_score(EMB_PCA, lbl))

best_k = ks[np.argmax(sils)]
print(f'Recommended K = {best_k}  (silhouette = {max(sils):.4f})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, sils, 'go-', lw=2, markersize=7)
ax1.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
ax1.set_title('Silhouette Score ↑', fontweight='bold'); ax1.set_xlabel('K'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ks, dbs, 'ro-', lw=2, markersize=7)
ax2.axvline(best_k, color='black', linestyle='--', label=f'Best K={best_k}')
ax2.set_title('Davies-Bouldin ↓', fontweight='bold'); ax2.set_xlabel('K'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('TS2Vec Embeddings — Optimal K Search', fontweight='bold')
plt.tight_layout(); plt.show()

# ── Cluster ───────────────────────────────────────────────────────────────
K = N_CLUSTERS   # or use best_k from above
kmeans = KMeans(n_clusters=K, n_init=30, random_state=42)
LABELS = kmeans.fit_predict(EMB_PCA)

print(f'\nCluster distribution (K={K}):')
u, c = np.unique(LABELS, return_counts=True)
for cluster, count in zip(u, c):
    bar = '█' * int(count / len(LABELS) * 40)
    print(f'  Surface {cluster}: {count:>4} windows ({count/len(LABELS)*100:5.1f}%)  {bar}')


## 7. Evaluation & Visualization <a id='7'></a>

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────
sil = silhouette_score(EMB_PCA, LABELS)
db  = davies_bouldin_score(EMB_PCA, LABELS)

quality = 'Excellent' if sil > 0.6 else 'Good' if sil > 0.4 else 'Fair' if sil > 0.2 else 'Poor'

print('TS2Vec Clustering Metrics')
print('─' * 35)
print(f'  Silhouette Score    : {sil:.4f}  ({quality})')
print(f'  Davies-Bouldin      : {db:.4f}')

if WINDOW_GT is not None:
    ari = adjusted_rand_score(WINDOW_GT, LABELS)
    print(f'  Adjusted Rand Index : {ari:.4f}  (vs ground truth)')


In [ ]:
# ── 2D Visualization ──────────────────────────────────────────────────────
print('Reducing to 2D for visualization...')

if UMAP_AVAILABLE:
    reducer = UMAP(n_components=2, metric='cosine', n_neighbors=30, min_dist=0.1, random_state=42)
    emb_2d  = reducer.fit_transform(EMB_NORM)
    dim_name = 'UMAP'
else:
    reducer = TSNE(n_components=2, metric='cosine', perplexity=30, random_state=42)
    emb_2d  = reducer.fit_transform(EMB_NORM)
    dim_name = 't-SNE'

n_plots = 2 if WINDOW_GT is not None else 1
fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 6))
if n_plots == 1: axes = [axes]
palette = plt.cm.tab10(np.linspace(0, 1, K))

# Predicted clusters
ax = axes[0]
for i in range(K):
    mask = LABELS == i
    ax.scatter(emb_2d[mask,0], emb_2d[mask,1], color=palette[i], s=15, alpha=0.7, label=f'Surface {i}')
ax.set_title(f'TS2Vec Embeddings — Predicted Clusters ({dim_name})', fontweight='bold')
ax.set_xlabel(f'{dim_name}-1'); ax.set_ylabel(f'{dim_name}-2')
ax.legend(markerscale=2); ax.grid(alpha=0.2)

# Ground truth
if WINDOW_GT is not None:
    ax2 = axes[1]
    gt_unique = np.unique(WINDOW_GT)
    gt_palette = plt.cm.Set2(np.linspace(0, 1, len(gt_unique)))
    for i, surf in enumerate(gt_unique):
        mask = np.array(WINDOW_GT) == surf
        ax2.scatter(emb_2d[mask,0], emb_2d[mask,1], color=gt_palette[i], s=15, alpha=0.7, label=surf)
    ax2.set_title(f'TS2Vec Embeddings — Ground Truth ({dim_name})', fontweight='bold')
    ax2.set_xlabel(f'{dim_name}-1'); ax2.set_ylabel(f'{dim_name}-2')
    ax2.legend(markerscale=2); ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# ── Timestamp-level: visualize surface transitions over time ──────────────
# Show how embedding changes as vehicle moves across surfaces

# Take first 5 windows across different surfaces
sample_windows = WINDOWS[:20]
emb_ts = model.encode(sample_windows, encoding_window=None, batch_size=BATCH_SIZE)
# emb_ts shape: (20, T, output_dims)

# Project timestamp embeddings to 1D using first PCA component
pca_1d = PCA(n_components=1)
ts_flat = emb_ts.reshape(-1, emb_ts.shape[-1])   # (20*T, dims)
ts_1d   = pca_1d.fit_transform(ts_flat).reshape(20, -1)   # (20, T)

fig, axes = plt.subplots(4, 5, figsize=(18, 8), sharex=True, sharey=True)
fig.suptitle('Timestamp-Level Embeddings (1st PCA dim) per Window', fontweight='bold')
for i, ax in enumerate(axes.flat):
    label = WINDOW_GT[i] if WINDOW_GT else f'Win {i}'
    ax.plot(ts_1d[i], lw=1.2, color='steelblue')
    ax.set_title(str(label), fontsize=8)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print('Each subplot = one window. Y-axis = embedding trajectory over time.')
print('Similar patterns across windows of the same surface = good representation ✅')


## 8. Fine-Tuning Tips for Better Embeddings <a id='8'></a>

### Problem → Solution guide

| Problem | Solution |
|---------|----------|
| Clusters overlap in UMAP | Increase `n_epochs` (try 100–200) |
| Loss not decreasing | Lower `lr` to 0.0005 |
| Out of memory (GPU/CPU) | Reduce `batch_size` to 4 or 2 |
| Too few distinct clusters | Increase `output_dims` (try 512) |
| Overfitting to one surface | Ensure balanced windows per surface |
| Speed still leaking into clusters | Increase `depth` to 12–15 |

### Recommended configs by dataset size

```python
# Small dataset (<2k windows)
model = TS2Vec(input_dims=3, output_dims=128, hidden_dims=64,  depth=6,  lr=0.001)
model.fit(X, n_epochs=100)

# Medium dataset (2k–10k windows)
model = TS2Vec(input_dims=3, output_dims=320, hidden_dims=64,  depth=10, lr=0.001)
model.fit(X, n_epochs=50)

# Large dataset (>10k windows)
model = TS2Vec(input_dims=3, output_dims=320, hidden_dims=128, depth=12, lr=0.0005)
model.fit(X, n_epochs=30)   # fewer epochs needed with more data
```

### Combine TS2Vec with Hand-Crafted (best of both worlds)
```python
emb_combined = np.hstack([
    normalize(EMB_INSTANCE),    # TS2Vec: learned patterns
    normalize(EMB_HC),          # Hand-crafted: physics features
])
# Then PCA + cluster as usual
```


## 9. Full Reusable Pipeline Function <a id='9'></a>

In [ ]:
def ts2vec_road_surface_pipeline(
    data,                        # np.array (N_total, 3) — denoised, z-scored
    fs             = 1000,
    window_sec     = 3.0,
    overlap        = 0.5,
    output_dims    = 320,
    n_epochs       = 50,
    n_clusters     = None,       # None = auto-detect
    gt_labels      = None,       # optional ground truth
    device         = 'cpu',
    model_save_path = 'ts2vec_model.pkl',
):
    """
    End-to-end: raw accelerometer → cluster labels.
    Returns dict with embeddings, labels, metrics.
    """
    # 1. Window
    win_len = int(fs * window_sec)
    step    = int(win_len * (1 - overlap))
    windows, win_gt = create_windows(data, win_len, step, gt_labels)
    print(f'[1] Windows: {windows.shape}')

    # 2. Train TS2Vec
    model = TS2Vec(input_dims=data.shape[1], output_dims=output_dims, device=device)
    model.fit(windows, n_epochs=n_epochs, verbose=False)
    model.save(model_save_path)
    print(f'[2] TS2Vec trained & saved → {model_save_path}')

    # 3. Encode (instance-level)
    emb = model.encode(windows, encoding_window='full_series')
    emb = np.nan_to_num(emb)
    print(f'[3] Embeddings: {emb.shape}')

    # 4. Post-process
    emb_norm = normalize(emb, norm='l2')
    emb_pca  = PCA(n_components=0.95, random_state=42).fit_transform(emb_norm)
    print(f'[4] PCA: {emb.shape[1]} → {emb_pca.shape[1]} dims')

    # 5. Auto K or use provided
    if n_clusters is None:
        sils = [silhouette_score(emb_pca,
                    KMeans(k, n_init=20, random_state=42).fit_predict(emb_pca))
                for k in range(2, 9)]
        n_clusters = list(range(2, 9))[np.argmax(sils)]
        print(f'[5] Auto K = {n_clusters} (silhouette={max(sils):.4f})')

    # 6. Cluster
    labels = KMeans(n_clusters=n_clusters, n_init=30, random_state=42).fit_predict(emb_pca)
    print(f'[6] Clustering done')

    # 7. Metrics
    metrics = {
        'silhouette':     silhouette_score(emb_pca, labels),
        'davies_bouldin': davies_bouldin_score(emb_pca, labels),
    }
    if win_gt is not None:
        metrics['ARI'] = adjusted_rand_score(win_gt, labels)
    print(f'[7] Silhouette={metrics["silhouette"]:.4f} | DB={metrics["davies_bouldin"]:.4f}')

    return {
        'model':      model,
        'windows':    windows,
        'embeddings': emb,
        'emb_norm':   emb_norm,
        'emb_pca':    emb_pca,
        'labels':     labels,
        'metrics':    metrics,
        'window_gt':  win_gt,
    }


# ── Run it ────────────────────────────────────────────────────────────────
results = ts2vec_road_surface_pipeline(
    data       = DATA,
    fs         = FS,
    window_sec = WINDOW_SEC,
    n_epochs   = N_EPOCHS,
    n_clusters = N_CLUSTERS,
    gt_labels  = GT_LABELS,
    device     = DEVICE,
)

print('\nDone ✅')
print(f'Cluster labels shape: {results["labels"].shape}')
print(f'Metrics: {results["metrics"]}')
